In [ ]:
#This notebook is for general tesing and debugging of the code
#You might notice each block is a module from "modular" folder

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [ ]:
#This is to test extraction and save everything for further testing
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

tickers = ["AAPL.US","MSFT.US","ABDE.US","AMZN.US","PEP.US"]
end=datetime.today()
start=end-timedelta(days=2*365)
flag=True
for ticker in tickers:
    url=f"https://stooq.com/q/d/l/?s={ticker}&d1={start:%Y%m%d}&d2={end:%Y%m%d}&i=d"
    df_temp=(pd.read_csv(url,parse_dates=["Date"])
        .set_index("Date")
        .sort_index())
    df_temp.columns = [f"{ticker} Open", f"{ticker} High", f"{ticker} Low", f"{ticker} Close", f"{ticker} Volume"]
    if flag:
        data=df_temp
        flag =False
    else:
        data=data.join(df_temp)

#Defining the period is one of the most important things and key to everything workin propperly
period=pd.Timedelta("2W")
df=data

In [ ]:
import pandas as pd
import pandas_datareader.data as web
from datetime import datetime, timedelta

tickers = ["AAPL.US", "MSFT.US", "ADBE.US", "AMZN.US", "PEP.US"] 
end = datetime.today()
start = end - timedelta(days=2*365)

# Descarga masiva: datareader acepta una lista y devuelve un DataFrame con MultiIndex
data = web.DataReader(tickers, 'stooq', start, end)

# El resultado viene con las fechas como índice y las columnas como MultiIndex:
# (Atributo, Ticker)
# Para dejarlo con el formato de nombres que querías ("Ticker Atributo"):

data.columns = [f"{ticker} {col}" for col, ticker in data.columns]
data = data.sort_index()



In [ ]:
df=data

In [ ]:
index="^NDX"
url=f"https://stooq.com/q/d/l/?s={index}&d1={start:%Y%m%d}&d2={end:%Y%m%d}&i=d"
data=(pd.read_csv(url,parse_dates=["Date"])
    .set_index("Date")
    .sort_index())
data.columns = [f"{index} Open", f"{index} High", f"{index} Low", f"{index} Close", f"{index} Volume"]

In [ ]:
index_name="^IPC"
index_v=index[f"{index_name} Close"].resample(period).mean()
    

In [ ]:
index_v

In [ ]:
data.to_csv("index_t.csv",index=True)

In [ ]:
df.to_csv("temporal.csv",index=True)

In [7]:
df_read = pd.read_csv("temporal.csv", index_col=0, parse_dates=True)
index=pd.read_csv("index_t.csv", index_col=0, parse_dates=True)


In [ ]:
#esto da la matriz de correlaciones en array
tickers = ["AAPL.US","MSFT.US"]
flag=True
for ticker in tickers:
    df_temp=df_read[f"{ticker} Close"]
    df_temp=pd.DataFrame(df_temp)
    if flag:
        data=df_temp
        flag =False
    else:
        data=data.join(df_temp)
data=data.corr().to_numpy()


In [ ]:
period=pd.Timedelta("2W")
valores=df_read[f"{tickers[j]} Close"].resample(period).mean()


In [ ]:
np.array(valores)
valor=[valores,valores]
np.array(valor)

In [ ]:
periods=53
portfolio=[]
for j in range(len(tickers)):
    portfolio.append([])
    valores=df_read[f"{tickers[j]} Close"].resample(period).mean()
    for i in range(periods):
        portfolio[j].append(valores.iloc[i])
print(portfolio)

In [ ]:
a = np.array([1, 2, 3, 4, 5])
b = np.array([2, 3, 4, 5, 6])
c = np.array([5, 4, 3, 2, 1])
d = np.array([1,2,3,4,5])
# Stack the arrays vertically into a single 2D array, where each row is a variable
# np.vstack() or passing a list of arrays works
data_stacked = np.vstack([a, b, c,d])

# Calculate the correlation matrix using np.corrcoef()
# The default is rowvar=True, meaning each row is a variable, which is what we need here
correlation_matrix_np = np.corrcoef(data_stacked)

In [ ]:
datos = [0]*5

datos[0]=[1,0.5,0,0,0]
datos[1]=[0.5,1,0,0,0]
datos[2]=[0,0,1,0.5,0.5]
datos[3]=[0,0,0.5,1,0]
datos[4]=[0,0,0.5,0,1]



In [ ]:
flag=True
for i in range(len(datos)):
    if flag:
        nuevo=datos[i]
        flag =False
    else:
        nuevo=np.vstack([nuevo,datos[i]])
nuevo=np.corrcoef(nuevo)

In [ ]:
nuevo

In [ ]:
def metric_correlation_matrix(matrix):
    for i in range(len(matrix)):
        for j in range(len(matrix)):
            matrix[i][j]=2*(1-matrix[i][j])
    print(matrix)
    matrix=np.sqrt(matrix)
    return matrix

In [ ]:
metric_correlation_matrix(nuevo)

In [ ]:
flag=True
for i in range(4):
    if flag:
        nuevo=data_stacked[i]
        flag =False
    else:
        nuevo=np.vstack([nuevo,data_stacked[i]])

In [ ]:
correlation_matrix_np

In [ ]:
#Para volver métricos
for i in range(len(data)):
    for j in range(len(data)):
        data[i][j]=2*(1-data[i][j])
np.sqrt(data)

In [ ]:
#Testing benchmark.py
def amount_of_periods(period,end,start):
    #from inputs import start_date as start
    #from inputs import end_date as end
    #from main_module import start 
    #from main_module import end
    num_periods = (end - start) / period
    round_num_periods=(end - start)//period
    if num_periods!=round_num_periods:
        round_num_periods=round_num_periods+1
    print(round_num_periods)
    return round_num_periods

def calc_dev_by_period(df,companies, period):
    deviations_by_period=[]
    for company in companies:
        deviation=(df[f"{company} High"] - df[f"{company} Low"]).resample(period).std()
        deviations_by_period.append(deviation)
        print(len(deviations_by_period[0]))
    return deviations_by_period

def calc_vola(df, companies, period,end,start):
    deviations_by_period=calc_dev_by_period(df,companies, period)
    num_periods=amount_of_periods(period,end,start)
    volatility_weight=[]
    for i in range(num_periods):
        volatility_weight.append([])
        desv_sum=0
        for j in range(len(companies)):
            deviations_by_period[j][i]=1/deviations_by_period[j][i]
            desv_sum=desv_sum+deviations_by_period[j][i]
        for j in range(len(companies)):
            volatility_weight[i].append(deviations_by_period[j][i]/desv_sum)
    return volatility_weight

In [ ]:
def index_value(index, period, index_name):
    index_v=index[f"{index_name} Close"].resample(period).mean()
    return index_v

def index_returns(index,period,index_name,end,start):
    periods=amount_of_periods(period,end,start)
    index_v=index_value(index, period, index_name)
    index_r=[0]*periods
    for i in range(periods-1):
        index_r[i]=(index_v[i+1]/index_v[i])-1
    return index_r

In [ ]:
end=datetime.today()
start=end-timedelta(days=2*365)
index_name="^IPC"

In [ ]:
index_returns(index,period,index_name,end,start)

In [ ]:
benchmark=calc_vola(df,tickers,period,end,start)

In [ ]:
#Testing portfolio .py
def portfolio_value(benchmark, df, period, tickers):
    periods=amount_of_periods(period, end, start)
    portfolio=[0]*periods
    for j in range(len(tickers)):
        valores=df[f"{tickers[j]} Close"].resample(period).mean()
        for i in range(periods):
            portfolio[i]=portfolio[i]+benchmark[i][j]*valores[i]
    return portfolio

def portfolio_returns(portfolio,period,end,start):
    periods=amount_of_periods(period,end,start)
    port_return=[0]*periods
    for i in range(periods-1):
        port_return[i]=(portfolio[i+1]/portfolio[i])-1
    return port_return

In [ ]:
port=portfolio_value(calc_vola(df,tickers,period, end, start),df,period,tickers)

In [ ]:
r=portfolio_returns(port,period,end,start)
print(r)

In [10]:
def amount_of_periods(period,start,end):
    num_periods = (end - start) / period
    round_num_periods=(end - start)//period
    if num_periods!=round_num_periods:
        round_num_periods=round_num_periods+1
    return round_num_periods

In [13]:
def general_portfolio_values(df, period, num_periods,tickers):
    portfolio=[]
    for j in range(len(tickers)):
        portfolio.append([])
        valores=df[f"{tickers[j]} Close"].resample(period).mean()
        for i in range(num_periods):
            portfolio[j].append(valores[i])
    return portfolio

In [42]:
def clusterized_portfolio(portfolio,medoids):
    cluster=[]
    for i in range(len(medoids)):
        cluster.append(portfolio[medoids[i]])
    return cluster

def index_value(index, period, index_name):
    index_v=index[f"{index_name} Close"].resample(period).mean()
    return index_v
def index_returns(index,period,num_periods,index_name):
    index_v=index_value(index, period, index_name)
    index_r=[0]*num_periods
    for i in range(num_periods-1):
        index_r[i]=(index_v[i+1]/index_v[i])-1
    return index_r
def general_portfolio_returns(portfolio,num_periods):
    r=[]
    for j in range(len(portfolio)):
        r.append([])
        for i in range(num_periods-1):
            r[j].append((portfolio[j][i+1]/portfolio[j][i])-1)
    return r

In [22]:
tickers = ["AAPL.US","MSFT.US","ADBE.US","AMZN.US","PEP.US"]
index="^NDX"
end=datetime(2026,3,15)
start=datetime(2024,3,17)
# por ahora por falla de stooq df=full_dataframe_extraction(tickers, start, end)
# index=index_dataframe_extraction(index, start, end)
df = pd.read_csv("C:/Users/praxy/OneDrive/Escritorio/Progra/Tests_for_live_index/temporal.csv", index_col=0, parse_dates=True)
index=pd.read_csv("C:/Users/praxy/OneDrive/Escritorio/Progra/Tests_for_live_index/index_t.csv", index_col=0, parse_dates=True)
period=pd.Timedelta("2W")
num_periods=amount_of_periods(period,start,end)

In [23]:
portfolio=general_portfolio_values(df,period,num_periods,tickers)
print(portfolio)

[[171.65177777777777, 169.0618, 166.9374, 176.9817, 188.1513, 191.9671111111111, 208.38477777777777, 214.1108888888889, 227.87060000000002, 218.54569999999998, 215.38180000000003, 225.79519999999997, 220.511, 223.38180000000003, 225.9333, 231.6129, 225.61610000000002, 226.0269, 237.9592222222222, 248.2723, 251.10312499999998, 235.66188888888888, 229.40433333333334, 232.8173, 242.99455555555556, 226.4824, 217.4625, 199.5259, 200.543, 203.523, 206.8765, 201.10666666666668, 198.49211111111111, 204.5048888888889, 209.9344, 210.8551, 222.0067, 228.956, 234.30399999999997, 246.99769999999998, 254.773, 254.7901, 269.4137, 270.471, 280.24555555555554, 275.658, 272.53125, 260.336, 253.17999999999998, 271.132, 267.0077777777778, 259.587], [420.82800000000003, 421.0236, 404.368, 402.8867, 421.9062, 419.653, 440.09644444444444, 453.26522222222223, 450.75969999999995, 423.78239999999994, 406.5114, 416.2668, 413.5035555555556, 431.67389999999995, 416.62510000000003, 420.4237, 418.7478, 418.033299999

C:\Users\praxy\AppData\Local\Temp\ipykernel_5464\4109720624.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  portfolio[j].append(valores[i])


In [33]:
medoids=[4,1]
optimo=clusterized_portfolio(portfolio,medoids)
print (optimo)

[[172.58888888888887, 169.849, 172.768, 176.79000000000002, 180.627, 172.30666666666667, 165.39111111111112, 165.20222222222222, 165.618, 171.794, 172.572, 174.66500000000002, 177.67666666666668, 172.625, 170.54500000000002, 174.249, 166.323, 161.324, 161.6211111111111, 156.38, 151.63375, 145.5411111111111, 150.24444444444444, 145.13, 151.08666666666667, 152.679, 148.06099999999998, 146.436, 140.96222222222224, 132.74650000000003, 130.712, 131.22777777777776, 130.4088888888889, 131.8422222222222, 136.872, 142.688, 144.905, 148.953, 145.26999999999998, 140.98499999999999, 142.21599999999998, 152.088, 145.644, 145.91000000000003, 147.38222222222223, 148.886, 144.05, 141.882, 147.64999999999998, 165.798, 166.6877777777778, 161.74200000000002], [420.82800000000003, 421.0236, 404.368, 402.8867, 421.9062, 419.653, 440.09644444444444, 453.26522222222223, 450.75969999999995, 423.78239999999994, 406.5114, 416.2668, 413.5035555555556, 431.67389999999995, 416.62510000000003, 420.4237, 418.7478, 4

In [43]:
optimo=general_portfolio_returns(optimo,num_periods)

In [39]:
index_name="^NDX"
index=index_returns(index,period,num_periods,index_name)

C:\Users\praxy\AppData\Local\Temp\ipykernel_5464\3486735890.py:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  index_r[i]=(index_v[i+1]/index_v[i])-1


In [ ]:
diferencia=[]
for i in range(len(optimo)):
    diferencia.append([])
    for j in range(num_periods):
        diferencia[i].append(index[j]-optimo[i][j])



IndexError: list index out of range